# 01 - Creación y Exploración de la Red Vial

Este notebook carga las redes viales de Punta Arenas (N505 y N1207) creadas con `NetworkCreator`,
calcula métricas de centralidad topológica y visualiza la estructura de la red.

**Pipeline:**
1. Cargar archivos `.dat` (posiciones, conexiones, carriles)
2. Construir grafos dirigidos y no dirigidos (intersecciones y calles)
3. Calcular métricas de centralidad (BC, CC, DC)
4. Calcular métricas de Random Walk / Ehrenfest
5. Visualizar la red con métricas superpuestas

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import scienceplots

# NetworkData se importa desde src/
from src.network.GetDataFromNetwork import NetworkData

: 

## 1. Cargar las redes viales

In [2]:
# Rutas de las redes
PATH_IMG = 'data/raw/Images/screenshotsGoogleMaps/screenshots/2023-03-29_07-30.png'
PATH_N505 = 'data/network/pickles/PuntaArenas'
PATH_N1207 = 'data/network/pickles/PuntaArenasDetallado'

# Cargar redes
PA_N505 = NetworkData(PATH_N505, PATH_IMG)
PA_N1207 = NetworkData(PATH_N1207, PATH_IMG)

NameError: name 'NetworkData' is not defined

In [ ]:
# Resumen de las redes
for label, net in [('N505', PA_N505), ('N1207', PA_N1207)]:
    print(f"\n{'='*40}")
    print(f"Red: {label}")
    print(f"  Intersecciones (nodos): {net.DG_inter.number_of_nodes()}")
    print(f"  Calles (aristas inter): {net.DG_inter.number_of_edges()}")
    print(f"  Conexiones entre calles: {len(net.conection_btw_streets)}")
    print(f"  Calles: {len(net.inter_conections)}")

## 2. Visualizar la estructura de la red

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, label, net in zip(axes, ['N505', 'N1207'], [PA_N505, PA_N1207]):
    ax.imshow(net.img_without_traffic)
    nx.draw_networkx_edges(net.DG_inter, net.pos_intersections, 
                           width=0.3, arrowsize=5, node_size=15, ax=ax)
    nx.draw_networkx_nodes(net.DG_inter, net.pos_intersections, 
                           node_size=10, node_color='red', ax=ax)
    ax.set_title(f'Red {label} - {net.DG_inter.number_of_nodes()} intersecciones', fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/figures/red_vial_estructura.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Calcular métricas de centralidad

In [ ]:
def compute_centrality_metrics(net, label):
    """Calcula todas las métricas de centralidad para una red."""
    data = {}
    
    # Centralidad de Betweenness (Dirigida y No Dirigida)
    data['DiBC'] = nx.betweenness_centrality(net.DG_streets)
    data['BC'] = nx.betweenness_centrality(net.UG_streets)
    
    # Centralidad de Closeness (Dirigida y No Dirigida)
    data['DiCC'] = nx.closeness_centrality(net.DG_streets)
    data['CC'] = nx.closeness_centrality(net.UG_streets)
    
    # Centralidad de Degree (Dirigida y No Dirigida)
    data['DiDC'] = nx.degree_centrality(net.DG_streets)
    data['DC'] = nx.degree_centrality(net.UG_streets)
    
    # Crear DataFrame
    df = net.makeDataFrame(data['DiBC'], 'DiBC')
    for col_name in ['BC', 'DiCC', 'CC', 'DiDC', 'DC']:
        df = net.addColumnToDataFrame(df, data[col_name], col_name)
    
    return df

df_N505 = compute_centrality_metrics(PA_N505, 'N505')
df_N1207 = compute_centrality_metrics(PA_N1207, 'N1207')

print("N505 - Primeras filas:")
display(df_N505.head())
print("\nN1207 - Primeras filas:")
display(df_N1207.head())

## 4. Distribución de métricas de centralidad

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
metricas = ['DiBC', 'BC', 'DiCC', 'CC', 'DiDC', 'DC']

for ax, metrica in zip(axes.flat, metricas):
    ax.hist(df_N505[metrica], bins=30, alpha=0.6, label='N505', color='blue')
    ax.hist(df_N1207[metrica], bins=30, alpha=0.6, label='N1207', color='red')
    ax.set_title(metrica, fontsize=14)
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.suptitle('Distribución de Métricas de Centralidad', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('results/figures/distribucion_centralidad.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Visualizar métricas sobre la red

In [ ]:
# Visualizar Closeness Centrality (Dirigida) para ambas redes
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, label, net, df in zip(axes, ['N505', 'N1207'], [PA_N505, PA_N1207], [df_N505, df_N1207]):
    ax.imshow(net.img_without_traffic)
    
    indices = list(df['DiCC'])
    indices_ordenado = [indices[i] for i in net.DG_inter.nodes()]
    
    nx.draw_networkx_edges(net.DG_inter, net.pos_intersections, width=0.3, arrowsize=5, 
                           node_size=15, ax=ax)
    nodes = nx.draw_networkx_nodes(net.DG_inter, net.pos_intersections, 
                                    node_size=30, node_color=indices_ordenado, 
                                    cmap=plt.cm.Reds, ax=ax)
    plt.colorbar(nodes, ax=ax, label='DiCC', shrink=0.8)
    ax.set_title(f'{label} - Closeness Centrality (Dirigida)', fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/figures/closeness_centrality_mapa.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Matrices de adyacencia

In [ ]:
adj_inter_N505, adj_streets_N505 = PA_N505.getAdjacencyMatrix()
adj_inter_N1207, adj_streets_N1207 = PA_N1207.getAdjacencyMatrix()

print(f"N505 - Matriz adyacencia intersecciones: {adj_inter_N505.shape}")
print(f"N505 - Matriz adyacencia calles: {adj_streets_N505.shape}")
print(f"N1207 - Matriz adyacencia intersecciones: {adj_inter_N1207.shape}")
print(f"N1207 - Matriz adyacencia calles: {adj_streets_N1207.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(adj_inter_N505, cmap='binary', interpolation='nearest')
axes[0].set_title('N505 - Adyacencia Intersecciones')
axes[1].imshow(adj_inter_N1207, cmap='binary', interpolation='nearest')
axes[1].set_title('N1207 - Adyacencia Intersecciones')
plt.tight_layout()
plt.savefig('results/figures/matriz_adyacencia.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Guardar datos de centralidad

In [ ]:
os.makedirs('data/processed/DataNetwork/StreetAsNode', exist_ok=True)

# Guardar si no existen (descomentar para ejecutar)
# df_N505.to_csv('data/processed/DataNetwork/StreetAsNode/all_data_SimpleNet_new.csv')
# df_N1207.to_csv('data/processed/DataNetwork/StreetAsNode/all_data_ComplexNet_new.csv')

print("Datos de centralidad listos para el siguiente paso del pipeline.")